# Tamil Palm-Leaf — Full Model Training

## ⚠️ IMPORTANT — Before Running:
1. Go to `Runtime → Change runtime type`
2. Select **T4 GPU** (NOT TPU)
3. Click Save
4. Then run all cells top to bottom

In [ ]:
# Cell 1 — Verify GPU (must show CUDA: True)
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('ERROR: No GPU found!')
    print('Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q torch torchvision opencv-python-headless Pillow numpy

In [ ]:
# Cell 3 — Upload your ZIP file
from google.colab import files
print('Upload archive (9).zip now...')
uploaded = files.upload()
ZIP_NAME = list(uploaded.keys())[0]
print(f'Uploaded: {ZIP_NAME}')

In [ ]:
# Cell 4 — Extract ZIP to disk (fixes the zlib worker error)
import zipfile, os

EXTRACT_DIR = '/content/tamil_data'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting ZIP... (this takes ~1 minute)')
with zipfile.ZipFile(ZIP_NAME, 'r') as z:
    z.extractall(EXTRACT_DIR)
print('Extraction complete!')

# Verify structure
from collections import Counter
counts = Counter()
for root, dirs, files_list in os.walk(EXTRACT_DIR):
    folder = os.path.basename(root)
    if folder.isdigit():
        counts[int(folder)] = len([f for f in files_list if f.endswith('.png')])

print(f'Classes found : {len(counts)}')
print(f'Total images  : {sum(counts.values())}')
print(f'Images/class  : ~{sum(counts.values())//max(len(counts),1)}')

In [ ]:
# Cell 5 — Build image path list from extracted files
import numpy as np

NUM_CLASSES = 71
IMG_SIZE    = 64
REST_SIZE   = 256

# Collect all (path, class_idx) pairs
all_samples = []
for root, dirs, files_list in os.walk(EXTRACT_DIR):
    folder = os.path.basename(root)
    if folder.isdigit():
        cls_idx = int(folder) - 1  # 0-based
        for f in files_list:
            if f.lower().endswith('.png'):
                all_samples.append((os.path.join(root, f), cls_idx))

print(f'Total samples: {len(all_samples)}')
print(f'Sample entry : {all_samples[0]}')

In [ ]:
# Cell 6 — Dataset classes (reads from extracted files — no ZIP worker issue)
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

def add_damage(img):
    d = img.copy()
    h, w = d.shape
    for _ in range(np.random.randint(2, 8)):
        cv2.line(d,
            (np.random.randint(0,w), np.random.randint(0,h)),
            (np.random.randint(0,w), np.random.randint(0,h)),
            0, np.random.randint(1, 3))
    for _ in range(np.random.randint(1, 4)):
        cv2.circle(d,
            (np.random.randint(0,w), np.random.randint(0,h)),
            np.random.randint(3, 12), 0, -1)
    noise = np.random.rand(h, w)
    d[noise < 0.04] = 0
    d[noise > 0.96] = 255
    return d


class TamilCharDataset(Dataset):
    def __init__(self, samples, split='train'):
        np.random.seed(42)
        idx = np.random.permutation(len(samples))
        cut = int(len(idx) * 0.85)
        chosen = idx[:cut] if split == 'train' else idx[cut:]
        self.samples = [samples[i] for i in chosen]
        self.tfm = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomRotation(10) if split=='train' else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        print(f'TamilCharDataset [{split}]: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = np.array(Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE)))
        return self.tfm(img), torch.tensor(label, dtype=torch.long)


class TamilRestoreDataset(Dataset):
    def __init__(self, samples, repeat=3):
        self.paths  = [s[0] for s in samples]
        self.repeat = repeat
        self.tt     = transforms.ToTensor()
        print(f'TamilRestoreDataset: {len(self.paths)} x {repeat} = {len(self)} samples')

    def __len__(self): return len(self.paths) * self.repeat

    def __getitem__(self, i):
        path  = self.paths[i % len(self.paths)]
        img   = np.array(Image.open(path).convert('L').resize((REST_SIZE, REST_SIZE)))
        # Paste on manuscript-like background
        bg    = np.ones((REST_SIZE, REST_SIZE), dtype=np.uint8) * 220
        off   = (REST_SIZE - img.shape[0]) // 2
        bg[off:off+img.shape[0], off:off+img.shape[1]] = img
        return self.tt(add_damage(bg)), self.tt(bg)


print('Dataset classes ready.')

In [ ]:
# Cell 7 — U-Net model
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
            nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(self, x): return self.b(x)

class UNet(nn.Module):
    def __init__(self, f=(32,64,128)):
        super().__init__()
        self.encs  = nn.ModuleList()
        self.pools = nn.ModuleList()
        c = 1
        for fi in f:
            self.encs.append(ConvBlock(c, fi))
            self.pools.append(nn.MaxPool2d(2))
            c = fi
        self.bot = ConvBlock(f[-1], f[-1]*2)
        self.ups  = nn.ModuleList()
        self.decs = nn.ModuleList()
        c = f[-1]*2
        for fi in reversed(f):
            self.ups.append(nn.ConvTranspose2d(c, fi, 2, stride=2))
            self.decs.append(ConvBlock(fi*2, fi))
            c = fi
        self.out = nn.Sequential(nn.Conv2d(f[0],1,1), nn.Sigmoid())

    def forward(self, x):
        skips = []
        for e, p in zip(self.encs, self.pools):
            x = e(x); skips.append(x); x = p(x)
        x = self.bot(x)
        for u, d, s in zip(self.ups, self.decs, reversed(skips)):
            x = d(torch.cat([u(x), s], 1))
        return self.out(x)

print('U-Net defined.')

In [ ]:
# Cell 8 — Tamil Char CNN (MobileNetV2)
from torchvision import models

class TamilCharCNN(nn.Module):
    def __init__(self, num_classes=71):
        super().__init__()
        base = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        old  = base.features[0][0]
        base.features[0][0] = nn.Conv2d(
            1, old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding, bias=False)
        with torch.no_grad():
            base.features[0][0].weight = nn.Parameter(
                old.weight.mean(dim=1, keepdim=True))
        self.features    = base.features
        self.pool        = nn.AdaptiveAvgPool2d((1,1))
        in_f             = base.classifier[1].in_features
        self.classifier  = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_f, 512), nn.ReLU(True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes))

    def forward(self, x):
        return self.classifier(torch.flatten(self.pool(self.features(x)), 1))

print('TamilCharCNN defined.')

In [ ]:
# Cell 9 — Train U-Net Restorer
import time

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS_R = 30
BATCH_R  = 16

restore_ds = TamilRestoreDataset(all_samples, repeat=3)
restore_dl = DataLoader(
    restore_ds, batch_size=BATCH_R, shuffle=True,
    num_workers=2,          # safe — reads from extracted files
    pin_memory=False        # avoids TPU/CPU warning
)

unet    = UNet().to(DEVICE)
opt_r   = torch.optim.Adam(unet.parameters(), lr=1e-3)
sched_r = torch.optim.lr_scheduler.CosineAnnealingLR(opt_r, T_max=EPOCHS_R)
crit_r  = nn.MSELoss()
best_r  = float('inf')
start   = time.time()

print(f'Training U-Net on {DEVICE} for {EPOCHS_R} epochs...')
for epoch in range(1, EPOCHS_R+1):
    unet.train()
    total = 0.0
    for dmg, cln in restore_dl:
        dmg, cln = dmg.to(DEVICE), cln.to(DEVICE)
        opt_r.zero_grad()
        loss = crit_r(unet(dmg), cln)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        opt_r.step()
        total += loss.item()
    sched_r.step()
    avg = total / len(restore_dl)
    if avg < best_r:
        best_r = avg
        torch.save(unet.state_dict(), 'restorer.pth')
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:02d}/{EPOCHS_R}]  Loss: {avg:.6f}  '
              f'Best: {best_r:.6f}  |  {time.time()-start:.0f}s')

print(f'U-Net done! Best loss: {best_r:.6f}  Saved: restorer.pth')

In [ ]:
# Cell 10 — Train Tamil Character CNN
EPOCHS_C = 30
BATCH_C  = 64

train_ds = TamilCharDataset(all_samples, split='train')
val_ds   = TamilCharDataset(all_samples, split='val')
train_dl = DataLoader(train_ds, batch_size=BATCH_C, shuffle=True,
                      num_workers=2, pin_memory=False)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_C, shuffle=False,
                      num_workers=2, pin_memory=False)

cnn     = TamilCharCNN(71).to(DEVICE)
opt_c   = torch.optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
sched_c = torch.optim.lr_scheduler.OneCycleLR(
    opt_c, max_lr=1e-3,
    steps_per_epoch=len(train_dl),
    epochs=EPOCHS_C, pct_start=0.1)
crit_c  = nn.CrossEntropyLoss(label_smoothing=0.1)
best_acc = 0.0
start    = time.time()

print(f'Training Tamil Char CNN on {DEVICE} for {EPOCHS_C} epochs...')
for epoch in range(1, EPOCHS_C+1):
    cnn.train()
    tloss = 0.0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        opt_c.zero_grad()
        loss = crit_c(cnn(imgs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
        opt_c.step()
        sched_c.step()
        tloss += loss.item()

    cnn.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            correct += (cnn(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)

    acc = correct / total if total > 0 else 0.0
    if acc > best_acc:
        best_acc = acc
        torch.save(cnn.state_dict(), 'tamil_char_model.pth')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:02d}/{EPOCHS_C}]  '
              f'Loss: {tloss/len(train_dl):.4f}  '
              f'Val: {acc*100:.1f}%  '
              f'Best: {best_acc*100:.1f}%  |  {time.time()-start:.0f}s')

print(f'CNN done! Best accuracy: {best_acc*100:.1f}%  Saved: tamil_char_model.pth')

In [ ]:
# Cell 11 — Visual check (before vs after restoration)
import matplotlib.pyplot as plt

unet.eval()
dmg_s, cln_s = restore_ds[0]
with torch.no_grad():
    res_s = unet(dmg_s.unsqueeze(0).to(DEVICE)).squeeze().cpu()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [dmg_s.squeeze(), res_s, cln_s.squeeze()],
    ['Damaged Input', 'Restored Output', 'Clean Target']):
    ax.imshow(img.numpy(), cmap='gray')
    ax.set_title(title, fontsize=13)
    ax.axis('off')
plt.suptitle('U-Net Restoration Result', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('restoration_preview.png', dpi=120)
plt.show()
print('Preview saved.')

In [ ]:
# Cell 12 — Download all model files
from google.colab import files
import os

for fname in ['restorer.pth', 'tamil_char_model.pth', 'restoration_preview.png']:
    if os.path.exists(fname):
        files.download(fname)
        size = os.path.getsize(fname) / (1024*1024)
        print(f'Downloaded: {fname}  ({size:.1f} MB)')
    else:
        print(f'Missing: {fname}')

print()
print('Next steps:')
print('  1. Move restorer.pth       -> palmleaf/models/restorer.pth')
print('  2. Move tamil_char_model.pth -> palmleaf/models/tamil_char_model.pth')
print('  3. Run: python app/app.py')